## 1 — OLS Regression: ε(t) ~ β·[SLP, T2m, u10, v10]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson

# ── Load København ──────────────────────────────────────────────────────
df = pd.read_parquet(
    "/home/nicaro/Python_project/data/per_station/station_30336_Kobenhavn.parquet"
)

df_subset = df[
    df["time"].between("2013-01-01", "2015-12-31")
]


# ── Target: model error ε(t) = FORCOAST − TG_obs ──────────────────────
df_subset["error_m"] = df_subset["forcoast_p82_m"] - df_subset["tg_obs_m"]

# ── Features: atmospheric forcings ─────────────────────────────────────
features = ["SLP", "t2m", "u10", "v10"]
feature_labels = ["SLP (msl)", "T2m", "u10", "v10"]

# Drop rows where any feature or target is NaN
df_clean = df_subset[features + ["error_m", "time", "tg_obs_m", "forcoast_p82_m"]].dropna() #dobbiamo usare 0 (di default) perche droppa tutta la riga. 
print(f"Total rows: {len(df_subset)}  →  Clean rows (no NaN): {len(df_clean)}")

X = df_clean[features].values
y = df_clean["error_m"].values

# Standardise features (zero mean, unit variance)
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X) # fit transfrom: per ogni colonna calcola media e std, poi opera (xij-media)/std

# ── OLS linear regression ───────────────────────────────────────────────
model = LinearRegression()
model.fit(X_scaled, y)

print(f"Coefficients (β): {model.coef_}")
print(f"Intercept (β₀): {model.intercept_}")

y_pred    = model.predict(X_scaled)
residuals = y - y_pred

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2   = r2_score(y, y_pred)
bias = float(y_pred.mean() - y.mean())
dw   = durbin_watson(residuals)   # Durbin-Watson statistic

print(f'somma nan: {df_subset[features + ["error_m"]].isna().sum()}')

# ── Print results ───────────────────────────────────────────────────────
print("\n═══════════════════════════════════════")
print("  Linear regression:  ε(t) ~ β·X(t)")
print("  Station: København")
print("═══════════════════════════════════════")
print(f"  Intercept (β₀)  : {model.intercept_:+.5f} m")
for name, coef in zip(feature_labels, model.coef_):
    print(f"  {name:12s} (β)  : {coef:+.5f} m/σ")
print("───────────────────────────────────────")
print(f"  R²              : {r2:.4f}")
print(f"  RMSE            : {rmse*100:.2f} cm")
print(f"  Bias (ŷ−y)      : {bias*100:+.8f} cm")
print(f"  Durbin-Watson   : {dw:.4f}   (2=no autocorr, <2=pos, >2=neg)")
print("═══════════════════════════════════════")

# ── Figure: 5 panels (2 top + 3 bottom) ────────────────────────────────
t = pd.to_datetime(df_clean["time"])

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "Linear Regression of Model Error — København\n"
    f"ε(t) = β·[SLP, T2m, u10, v10]   |   R² = {r2:.3f}   "
    f"RMSE = {rmse*100:.2f} cm   DW = {dw:.2f}",
    fontsize=13,
)

# Layout: 2 righe — top ha 2 pannelli, bottom ha 3
gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.32)

# ── Panel 1 (top-left span 2 cols): ε(t) observed vs predicted ──────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(t, y,      color="#2980B9", lw=0.5, alpha=0.7, label="ε observed")
ax1.plot(t, y_pred, color="#E74C3C", lw=0.8, alpha=0.9, label="ε predicted (OLS)")
ax1.axhline(0, color="grey", lw=0.5, ls="--")
ax1.set_title("Error time series")
ax1.set_ylabel("ε [m]")
ax1.legend(fontsize=8)

# ── Panel 2 (top-right): scatter ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
ax2.scatter(y, y_pred, s=1, alpha=0.2, color="#8E44AD")
ax2.plot(lims, lims, "k--", lw=0.8)
ax2.set_xlabel("ε observed [m]")
ax2.set_ylabel("ε predicted [m]")
ax2.set_title(f"Scatter (R² = {r2:.3f})")

# ── Panel 3 (bottom-left): residuals over time ──────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(t, residuals, color="#27AE60", lw=0.4, alpha=0.7)
ax3.axhline(0, color="grey", lw=0.5, ls="--")
ax3.set_title(f"Residuals  (RMSE = {rmse*100:.2f} cm)")
ax3.set_ylabel("residual [m]")
ax3.set_xlabel("Time")

# ── Panel 4 (bottom-centre): standardised β bar chart ───────────────────
ax4 = fig.add_subplot(gs[1, 1])
colors = ["#E74C3C" if c < 0 else "#2980B9" for c in model.coef_]
ax4.barh(feature_labels, model.coef_, color=colors, edgecolor="white")
ax4.axvline(0, color="grey", lw=0.8)
ax4.set_title("Standardised coefficients β")
ax4.set_xlabel("β [m/σ]")

# ── Panel 5 (bottom-right): ACF of residuals ────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])

# Numero di lag: max 168 (1 settimana se dati orari) o 10% della serie
n_lags = min(168, len(residuals) // 10)

plot_acf(
    residuals,
    lags=n_lags,
    ax=ax5,
    alpha=0.05,           # banda di confidenza al 95%
    color="#E67E22",
    vlines_kwargs={"lw": 0.5},
    zero=False,           # non mostrare il lag 0 (sempre = 1)
    title=""
)
ax5.set_title(f"ACF of residuals  (DW = {dw:.2f})")
ax5.set_xlabel("Lag [hours]")
ax5.set_ylabel("Autocorrelation")
ax5.axhline(0, color="grey", lw=0.5, ls="--")

# Aggiungi linee verticali per periodicità notevoli (dati orari)
for lag, label in [(12, "12h"), (24, "24h"), (48, "48h")]:
    if lag <= n_lags:
        ax5.axvline(lag, color="red", lw=0.7, ls=":", alpha=0.6)
        ax5.text(lag, ax5.get_ylim()[1]*0.85, label,
                 fontsize=7, color="red", ha="center")

plt.show()


## 2 — MISO Lag Regression (L = 72 h)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson

# ════════════════════════════════════════════════════════════════════════
#  USER PARAMETERS — CASO CON LAG
# ════════════════════════════════════════════════════════════════════════
L_lag          = 2400
STATION_lag    = "København"
PARQUET_lag    = "/home/nicaro/Python_project/data/per_station/station_30336_Kobenhavn.parquet"
FEATURES_lag   = ["SLP", "t2m", "u10", "v10"]
F_LABELS_lag   = ["SLP", "T2m", "u10", "v10"]
TARGET_COL_lag = "error_m"
# ════════════════════════════════════════════════════════════════════════

# 1) Load & compute error
df_lag_source = pd.read_parquet(PARQUET_lag)
df_subset_lag = df_lag_source[df_lag_source["time"].between("2013-01-01", "2015-12-31")].copy()
df_subset_lag[TARGET_COL_lag] = df_subset_lag["forcoast_p82_m"] - df_subset_lag["tg_obs_m"]

# 2) Build lagged feature matrix
print(
    f"Building lag matrix: {len(FEATURES_lag)} features × {L_lag+1} lags = "
    f"{len(FEATURES_lag)*(L_lag+1)} columns ..."
)

lagged_columns_dict_lag = {}
for feat_lag in FEATURES_lag:
    for k_lag in range(0, L_lag + 1):
        col_name_lag = f"{feat_lag}_lag{k_lag:03d}"
        lagged_columns_dict_lag[col_name_lag] = df_subset_lag[feat_lag].shift(k_lag)

df_lag_features_lag = pd.DataFrame(lagged_columns_dict_lag)

# 3) Assemble model dataset and drop NaN
df_model_lag = pd.concat(
    [df_subset_lag[["time", TARGET_COL_lag]], df_lag_features_lag], axis=1
).dropna()

print(
    f"Rows after dropping NaN from lags: {len(df_model_lag)}  "
    f"(dropped first {L_lag} rows as expected)"
)

y_lag = df_model_lag[TARGET_COL_lag].values
X_raw_lag = df_model_lag[df_lag_features_lag.columns].values

# 4) Standardise
scaler_lag = StandardScaler()
X_scaled_lag = scaler_lag.fit_transform(X_raw_lag)

# 5) OLS
model_lag = LinearRegression()
model_lag.fit(X_scaled_lag, y_lag)
y_pred_lag = model_lag.predict(X_scaled_lag)
residuals_lag = y_lag - y_pred_lag



rmse_lag = np.sqrt(mean_squared_error(y_lag, y_pred_lag))
r2_lag = r2_score(y_lag, y_pred_lag)
dw_lag = durbin_watson(residuals_lag)

print("\n═══════════════════════════════════════════════")
print(f"  MISO Lag Regression — {STATION_lag}  (L = {L_lag}h)")
print("═══════════════════════════════════════════════")
print(f"  N features (total) : {X_scaled_lag.shape[1]}")
print(f"  R²                 : {r2_lag:.4f}")
print(f"  RMSE               : {rmse_lag*100:.2f} cm")
print(f"  Durbin-Watson      : {dw_lag:.4f}")
print("═══════════════════════════════════════════════")

# 6) Impulse Response Functions (IRF)
n_feat_lag = len(FEATURES_lag)
n_lags_lag = L_lag + 1
beta_matrix_lag = model_lag.coef_.reshape(n_feat_lag, n_lags_lag)

# 7) Figure
t_lag = pd.to_datetime(df_model_lag["time"])

fig_lag = plt.figure(figsize=(18, 12))
fig_lag.suptitle(
    f"MISO Lag Regression — {STATION_lag}   (L = {L_lag} h)\n"
    f"ε(t) = β₀ + Σ_k Σ_f β_{{f,k}}·f(t−k)   |   "
    f"R² = {r2_lag:.3f}   RMSE = {rmse_lag*100:.2f} cm   DW = {dw_lag:.2f}",
    fontsize=13,
)

gs_lag = fig_lag.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# Panel 1
ax1_lag = fig_lag.add_subplot(gs_lag[0, :])
ax1_lag.plot(t_lag, y_lag, color="#2980B9", lw=0.5, alpha=0.6, label="ε observed")
ax1_lag.plot(t_lag, y_pred_lag, color="#E74C3C", lw=0.8, alpha=0.9, label="ε predicted (MISO lag)")
ax1_lag.axhline(0, color="grey", lw=0.5, ls="--")
ax1_lag.set_title("Error time series (lag case)")
ax1_lag.set_ylabel("ε [m]")
ax1_lag.legend(fontsize=9)

# Panel 2
ax2_lag = fig_lag.add_subplot(gs_lag[1, 0])
lims_lag = [min(y_lag.min(), y_pred_lag.min()), max(y_lag.max(), y_pred_lag.max())]
ax2_lag.scatter(y_lag, y_pred_lag, s=1, alpha=0.15, color="#8E44AD")
ax2_lag.plot(lims_lag, lims_lag, "k--", lw=0.8)
ax2_lag.set_xlabel("ε observed [m]")
ax2_lag.set_ylabel("ε predicted [m]")
ax2_lag.set_title(f"Scatter  (R² = {r2_lag:.3f})")

# Panel 3
ax3_lag = fig_lag.add_subplot(gs_lag[1, 1])
ax3_lag.plot(t_lag, residuals_lag, color="#27AE60", lw=0.4, alpha=0.7)
ax3_lag.axhline(0, color="grey", lw=0.5, ls="--")
ax3_lag.set_title(f"Residuals  (RMSE = {rmse_lag*100:.2f} cm)")
ax3_lag.set_ylabel("residual [m]")
ax3_lag.set_xlabel("Time")

# Panel 4
ax4_lag = fig_lag.add_subplot(gs_lag[1, 2])
n_lags_acf_lag = min(168, len(residuals_lag) // 10)
plot_acf(
    residuals_lag,
    lags=n_lags_acf_lag,
    ax=ax4_lag,
    alpha=0.05,
    color="#E67E22",
    vlines_kwargs={"lw": 0.5},
    zero=False,
    title="",
)
ax4_lag.set_title(f"ACF residuals  (DW = {dw_lag:.2f})")
ax4_lag.set_xlabel("Lag [hours]")
ax4_lag.set_ylabel("Autocorrelation")
for lag_mark_lag, label_mark_lag in [(12, "12h"), (24, "24h"), (48, "48h")]:
    if lag_mark_lag <= n_lags_acf_lag:
        ax4_lag.axvline(lag_mark_lag, color="red", lw=0.7, ls=":", alpha=0.6)
        ax4_lag.text(
            lag_mark_lag,
            0.85,
            label_mark_lag,
            fontsize=7,
            color="red",
            ha="center",
            transform=ax4_lag.get_xaxis_transform(),
        )

# Panels 5–8: IRF
lags_x_lag = np.arange(0, L_lag + 1)
colors_irf_lag = ["#E74C3C", "#3498DB", "#2ECC71", "#F39C12"]

for i_lag, (feat_name_lag, feat_label_lag, color_lag) in enumerate(zip(FEATURES_lag, F_LABELS_lag, colors_irf_lag)):
    if i_lag < 3:
        ax_irf_lag = fig_lag.add_subplot(gs_lag[2, i_lag])
    else:
        ax_irf_lag = fig_lag.add_axes([0.38, 0.02, 0.27, 0.19])

    ax_irf_lag.plot(lags_x_lag, beta_matrix_lag[i_lag], color=color_lag, lw=1.5)
    ax_irf_lag.fill_between(lags_x_lag, beta_matrix_lag[i_lag], alpha=0.2, color=color_lag)
    ax_irf_lag.axhline(0, color="grey", lw=0.7, ls="--")
    ax_irf_lag.set_title(f"IRF — {feat_label_lag}")
    ax_irf_lag.set_xlabel("Lag k [hours]")
    ax_irf_lag.set_ylabel("β [m/σ]")

plt.show()

## 3 — Detiding (utide) + MISO Regression Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
import utide

# ════════════════════════════════════════════════════════════════════════
#  USER PARAMETERS  ← cambia qui
# ════════════════════════════════════════════════════════════════════════
L        = 72        # numero di lag orari (es. 72 = 3 giorni)
LAT      = 55.7      # latitudine di København
STATION  = "København"
PARQUET  = "/home/nicaro/Python_project/data/per_station/station_30336_Kobenhavn.parquet"
FEATURES = ["SLP", "t2m", "u10", "v10"]
F_LABELS = ["SLP", "T2m", "u10", "v10"]
CUTOFF   = "2015-12-31 23:00:00"   # ← usa dati solo fino a fine 2015
# ════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — Carica il dataset (troncato al 2015)                      ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("=" * 60)
print("STEP 1 — Loading data")
print("=" * 60)

df_subset = pd.read_parquet(PARQUET)
df_subset = df_subset.sort_values("time").reset_index(drop=True)
df_subset = df_subset[df_subset["time"] <= CUTOFF].reset_index(drop=True)

print(f"  Rows loaded       : {len(df_subset)}")
print(f"  Time range        : {df_subset['time'].min()} → {df_subset['time'].max()}")
print(f"  NaN in tg_obs_m   : {df_subset['tg_obs_m'].isna().sum()}")
print(f"  NaN in forcoast   : {df_subset['forcoast_p82_m'].isna().sum()}")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — Converti il tempo in float GIORNI per utide               ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
#  utide si aspetta il tempo in GIORNI (formato MATLAB datenum):
#    - t deve essere numpy 1D float (giorni da un'epoca)
#    - u deve essere numpy 1D float
#    - t e u devono avere ESATTAMENTE la stessa shape

print("\n" + "=" * 60)
print("STEP 2 — Preparing time array for utide  (DAYS)")
print("=" * 60)

t0       = df_subset["time"].min()    # epoca locale → t parte da 0
time_all = (df_subset["time"] - t0).dt.total_seconds().values / 86400.0  # GIORNI, float64

# Maschera: solo righe con TG_obs non-NaN
mask_tg = df_subset["tg_obs_m"].notna().values

# Applica la STESSA maschera a t e u → shape identica
t_fit = time_all[mask_tg]
u_fit = df_subset["tg_obs_m"].values[mask_tg]

print(f"  t_fit shape : {t_fit.shape}  (giorni da {t0})")
print(f"  u_fit shape : {u_fit.shape}")
print(f"  Time range  : {t_fit[0]:.2f} d → {t_fit[-1]:.2f} d  "
      f"({t_fit[-1]:.0f} giorni ≈ {t_fit[-1]/365.25:.1f} anni)")
assert t_fit.shape == u_fit.shape, "ERRORE: t e u hanno shape diverse!"


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — Fit armonico con utide                                    ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("STEP 3 — Harmonic analysis with utide")
print("=" * 60)

coef = utide.solve(
    t_fit,
    u_fit,
    lat=LAT,
    method="ols",
    conf_int="linear",
    verbose=True
)

# Stampa top 10 costituenti per ampiezza
df_const = pd.DataFrame({
    "name"      : coef.name,
    "amplitude" : coef.A,
    "phase_deg" : coef.g
}).sort_values("amplitude", ascending=False)

print("\n  Top 10 tidal constituents fitted on TG_obs:")
print(df_const.head(10).to_string(index=False))

# Controlla M2 specificamente
if "M2" in df_const["name"].values:
    m2 = df_const[df_const["name"] == "M2"].iloc[0]
    print(f"\n  ✅ M2 found:  amplitude = {m2['amplitude']:.4f} m   "
          f"phase = {m2['phase_deg']:.1f}°")
else:
    print("\n  ⚠️  M2 not found — check verbose output above")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 — Ricostruisce la marea su TUTTI i timestep                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("STEP 4 — Reconstructing tidal signal on all timesteps")
print("=" * 60)

# Ricostruisce su time_all (giorni) → include anche i gap del dataset
tide_fit    = utide.reconstruct(time_all, coef, verbose=False)
tide_signal = tide_fit.h    # numpy array 1D, shape = (len(df),)

print(f"  Tide signal  —  mean : {np.nanmean(tide_signal):+.4f} m")
print(f"                  std  : {np.nanstd(tide_signal):.4f} m")
print(f"                  range: [{tide_signal.min():.3f}, {tide_signal.max():.3f}] m")
print(f"\n  ✅ OK se std ~ 0.02–0.15 m (Baltico ha maree piccole)")
print(f"  ⚠️  BUG se std ~ 0 o se tide_signal è una retta")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 5 — Sottrai la marea e ricalcola gli errori                   ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("STEP 5 — Computing errors")
print("=" * 60)

df_subset["tide_utide_m"]   = tide_signal
df_subset["tg_notide_m"]    = df_subset["tg_obs_m"]       - tide_signal   # TG senza marea
df_subset["error_raw_m"]    = df_subset["forcoast_p82_m"] - df_subset["tg_obs_m"]      # ε totale
df_subset["error_notide_m"] = df_subset["forcoast_p82_m"] - df_subset["tg_notide_m"]   # ε senza marea

std_raw  = df_subset["error_raw_m"].std()
std_nd   = df_subset["error_notide_m"].std()

print(f"  ε_raw    std : {std_raw:.4f} m")
print(f"  ε_notide std : {std_nd:.4f} m")
if std_nd < std_raw:
    print(f"  ✅ Detiding riduce la varianza dell'errore di "
          f"{(std_raw - std_nd)*100:.2f} cm")
else:
    print(f"  ⚠️  ε_notide std > ε_raw std — controlla il detiding!")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 6 — Costruisci matrice dei lag (causalità rispettata)         ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print(f"STEP 6 — Building lag matrix  (L = {L} h)")
print("=" * 60)

lag_cols = {}
for feat in FEATURES:
    for k in range(0, L + 1):
        lag_cols[f"{feat}_lag{k:03d}"] = df_subset[feat].shift(k)

df_lag   = pd.DataFrame(lag_cols)
df_model = pd.concat(
    [df_subset[["time", "error_notide_m", "error_raw_m"]], df_lag], axis=1
).dropna()

print(f"  Lag matrix shape  : {df_lag.shape[1]} cols  "
      f"({len(FEATURES)} features × {L+1} lags)")
print(f"  Rows after dropna : {len(df_model)}  "
      f"(prime {L} righe droppate per costruzione lag)")

y_notide = df_model["error_notide_m"].values
y_raw    = df_model["error_raw_m"].values
X_raw    = df_model[df_lag.columns].values


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 7 — OLS baseline su ε_raw                                     ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("STEP 7 — OLS on ε_raw  (baseline)")
print("=" * 60)

scaler_raw = StandardScaler()
X_sc_raw   = scaler_raw.fit_transform(X_raw)
model_raw  = LinearRegression().fit(X_sc_raw, y_raw)
yp_raw     = model_raw.predict(X_sc_raw)
res_raw    = y_raw - yp_raw

r2_raw   = r2_score(y_raw, yp_raw)
rmse_raw = np.sqrt(mean_squared_error(y_raw, yp_raw))
dw_raw   = durbin_watson(res_raw)
print(f"  R² = {r2_raw:.4f}   RMSE = {rmse_raw*100:.2f} cm   DW = {dw_raw:.3f}")


# ╔══════════════════════════════════════════════════════════════════════╗
# ║  STEP 8 — OLS su ε_notide                                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
print("\n" + "=" * 60)
print("STEP 8 — OLS on ε_notide  (after detiding)")
print("=" * 60)

scaler_nd  = StandardScaler()
X_sc_nd    = scaler_nd.fit_transform(X_raw)
model_nd   = LinearRegression().fit(X_sc_nd, y_notide)
yp_notide  = model_nd.predict(X_sc_nd)
res_notide = y_notide - yp_notide

r2_nd   = r2_score(y_notide, yp_notide)
rmse_nd = np.sqrt(mean_squared_error(y_notide, yp_notide))
dw_nd   = durbin_watson(res_notide)
print(f"  R² = {r2_nd:.4f}   RMSE = {rmse_nd*100:.2f} cm   DW = {dw_nd:.3f}")

print("\n" + "─" * 40)
print(f"  ΔR²   = {r2_nd  - r2_raw:+.4f}   (notide − raw)")
print(f"  ΔRMSE = {(rmse_nd - rmse_raw)*100:+.2f} cm")
print(f"  ΔDW   = {dw_nd  - dw_raw:+.3f}")
print("─" * 40)


# ════════════════════════════════════════════════════════════════════════
#  FIGURE
# ════════════════════════════════════════════════════════════════════════
t_plot     = pd.to_datetime(df_model["time"])
t_full     = pd.to_datetime(df_subset["time"])
lags_x     = np.arange(0, L + 1)
beta_nd    = model_nd.coef_.reshape(len(FEATURES), L + 1)
beta_raw   = model_raw.coef_.reshape(len(FEATURES), L + 1)
colors_irf = ["#E74C3C", "#3498DB", "#2ECC71", "#F39C12"]
n_lags_acf = min(168, len(res_notide) // 10)


# ── FIGURA 1: Verifica detiding ─────────────────────────────────────────
fig1, ax1 = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig1.suptitle(f"Detiding verification — {STATION}  (2013–2015)", fontsize=13)

ax1[0].plot(t_full, df_subset["tg_obs_m"],    color="#2980B9", lw=0.5,
            alpha=0.8, label="TG obs (raw)")
ax1[0].plot(t_full, df_subset["tg_notide_m"], color="#E74C3C", lw=0.5,
            alpha=0.8, label="TG notide")
ax1[0].set_ylabel("Sea level [m]"); ax1[0].legend(fontsize=8)
ax1[0].set_title("TG_obs vs TG_notide")

ax1[1].plot(t_full, tide_signal, color="#8E44AD", lw=0.6, label="Fitted tide")
ax1[1].set_ylabel("Tide [m]"); ax1[1].legend(fontsize=8)
ax1[1].set_title("Fitted tidal signal — utide")

ax1[2].plot(t_full, df_subset["error_raw_m"],    color="#2980B9", lw=0.5, alpha=0.6,
            label=f"ε raw    (std={std_raw:.4f} m)")
ax1[2].plot(t_full, df_subset["error_notide_m"], color="#E74C3C", lw=0.5, alpha=0.7,
            label=f"ε notide (std={std_nd:.4f} m)")
ax1[2].axhline(0, color="grey", lw=0.5, ls="--")
ax1[2].set_ylabel("Error [m]"); ax1[2].set_xlabel("Time")
ax1[2].legend(fontsize=8)
ax1[2].set_title("ε_raw vs ε_notide")

plt.tight_layout()
plt.show()


# ── FIGURA 2: RAW vs NOTIDE — confronto regressione ────────────────────
fig2, axes2 = plt.subplots(3, 2, figsize=(16, 12))
fig2.suptitle(
    f"Detiding effect on MISO regression — {STATION}  (L = {L} h, 2013–2015)\n"
    f"LEFT: ε_raw  (R²={r2_raw:.3f}, DW={dw_raw:.2f})   |   "
    f"RIGHT: ε_notide  (R²={r2_nd:.3f}, DW={dw_nd:.2f})",
    fontsize=12
)

for ci, (y_, yp_, res_, lbl_, r2_, dw_) in enumerate([
    (y_raw,    yp_raw,    res_raw,    "ε raw",    r2_raw, dw_raw),
    (y_notide, yp_notide, res_notide, "ε notide", r2_nd,  dw_nd),
]):
    ax = axes2[0, ci]
    ax.plot(t_plot, y_,  color="#2980B9", lw=0.5, alpha=0.6, label="observed")
    ax.plot(t_plot, yp_, color="#E74C3C", lw=0.8, alpha=0.9, label="predicted")
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_title(f"{lbl_} — time series"); ax.set_ylabel("ε [m]")
    ax.legend(fontsize=8)

    ax = axes2[1, ci]
    ax.plot(t_plot, res_, color="#27AE60", lw=0.4, alpha=0.7)
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    rmse_ = np.sqrt(mean_squared_error(y_, yp_))
    ax.set_title(f"Residuals  RMSE = {rmse_*100:.2f} cm")
    ax.set_ylabel("residual [m]"); ax.set_xlabel("Time")

    ax = axes2[2, ci]
    plot_acf(res_, lags=n_lags_acf, ax=ax, alpha=0.05,
             color="#E67E22", vlines_kwargs={"lw": 0.5}, zero=False, title="")
    ax.set_title(f"ACF residuals  DW = {dw_:.2f}")
    ax.set_xlabel("Lag [hours]"); ax.set_ylabel("Autocorrelation")
    for lag, lbl in [(12, "12h"), (24, "24h"), (48, "48h")]:
        if lag <= n_lags_acf:
            ax.axvline(lag, color="red", lw=0.7, ls=":", alpha=0.6)
            ax.text(lag, 0.85, lbl, fontsize=7, color="red", ha="center",
                    transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.show()


# ── FIGURA 3: Impulse Response Functions — notide ───────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 8))
fig3.suptitle(
    f"Impulse Response Functions (IRF) — {STATION}  (L = {L} h, 2013–2015)\n"
    f"ε_notide model  |  R² = {r2_nd:.3f}   RMSE = {rmse_nd*100:.2f} cm   "
    f"DW = {dw_nd:.2f}",
    fontsize=12
)
axes3 = axes3.flatten()

for i, (feat, label, col) in enumerate(zip(FEATURES, F_LABELS, colors_irf)):
    ax = axes3[i]
    ax.plot(lags_x, beta_nd[i],  color=col,    lw=1.8, label="ε_notide")
    ax.plot(lags_x, beta_raw[i], color="grey", lw=1.0, ls="--",
            alpha=0.6, label="ε_raw (baseline)")
    ax.fill_between(lags_x, beta_nd[i], alpha=0.15, color=col)
    ax.axhline(0, color="grey", lw=0.7, ls="--")

    k_max = int(np.argmax(np.abs(beta_nd[i])))
    ax.axvline(k_max, color=col, lw=1.0, ls=":", alpha=0.8)
    ax.text(k_max + 1, beta_nd[i][k_max], f"peak @ {k_max}h",
            fontsize=8, color=col)

    ax.set_title(f"IRF — {label}")
    ax.set_xlabel("Lag k [hours]")
    ax.set_ylabel("β [m/σ]")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ── FIGURA 4: Scatter ε_notide osservato vs predetto ────────────────────
fig4, ax4 = plt.subplots(1, 1, figsize=(6, 6))
lims = [min(y_notide.min(), yp_notide.min()),
        max(y_notide.max(), yp_notide.max())]
ax4.scatter(y_notide, yp_notide, s=1, alpha=0.15, color="#8E44AD")
ax4.plot(lims, lims, "k--", lw=0.8, label="1:1")
ax4.set_xlabel("ε_notide observed [m]")
ax4.set_ylabel("ε_notide predicted [m]")
ax4.set_title(f"Scatter — ε_notide  (R² = {r2_nd:.3f})")
ax4.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\n{'═'*50}")
print(f"  FINAL SUMMARY — {STATION}  (L = {L} h, 2013–2015)")
print(f"{'═'*50}")
print(f"  ε_raw    →  R² = {r2_raw:.4f}  RMSE = {rmse_raw*100:.2f} cm  DW = {dw_raw:.3f}")
print(f"  ε_notide →  R² = {r2_nd:.4f}  RMSE = {rmse_nd*100:.2f} cm  DW = {dw_nd:.3f}")
print(f"  ΔR²      = {r2_nd - r2_raw:+.4f}")
print(f"  ΔRMSE    = {(rmse_nd - rmse_raw)*100:+.2f} cm")
print(f"  ΔDW      = {dw_nd - dw_raw:+.3f}")
print(f"{'═'*50}")